# V4 · Thời gian, thời tiết và lịch sử lưu lượng

**Mục tiêu:** đo giá trị bổ sung của lịch sử (đối chứng B–C) và đánh giá baseline seasonal naive, baseline còn thiếu trong ba baseline của proposal.

**Kịch bản:** dự đoán tuần tự từng giờ. Tại giờ `t`, các quan sát trước `t` đã được ghi nhận, kể cả các giờ validation trước đó. Đây không phải dự đoán đồng thời toàn bộ tương lai.

| Đặc trưng | Cách tính |
|---|---|
| `lag_24h` | Lưu lượng tại `t − 24 giờ`, tra theo timestamp |
| `lag_168h` | Lưu lượng tại `t − 168 giờ`, tra theo timestamp |
| `mean_24h` | Trung bình các giờ quan sát trong `[t − 24 giờ, t)`, cần ít nhất 18 giờ |

**Chạy trước:** `V_3.ipynb`. Test giữ riêng.

**Tuning theo trang 4.** Notebook hiển thị mốc fold và bảng MAE từng ứng viên. `src/tuning.py` thực hiện chia fold và tìm tham số. Với cấu hình thời tiết, mỗi fold dùng quan sát gốc để học median từ phần thời gian trước validation. LR là mốc không tuning. RF, GB và MLP theo sơ đồ, Ridge và XGBoost giữ để đối chứng. Kết quả đã lưu trong notebook là kết quả lịch sử. Khi chạy lại V2 → V3 → V4, kết quả mới lưu trong `experiments/experiments_logs.xlsx`.

## 4.1 · Chuẩn bị dữ liệu

In [10]:
from pathlib import Path
import json
import os
import platform
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import xgboost

# Tìm gốc dự án từ thư mục hiện tại hoặc các thư mục cha.
current_dir = Path.cwd().resolve()
project_dir = next(
    (p for p in (current_dir, *current_dir.parents)
     if (p / "data" / "Metro_Interstate_Traffic_Volume.csv").is_file()),
    None,
)
if project_dir is None:
    raise FileNotFoundError("Không tìm thấy CSV trong thư mục dự án.")
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

output_dir = project_dir / "experiments"
figure_dir = project_dir / "figures"
pd.set_option("display.max_columns", None)
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e5e4e0", "axes.axisbelow": True,
    "lines.linewidth": 2,
})
BLUE, ORANGE = "#2a78d6", "#eb6834"

print("Python:", platform.python_version())
print("pandas:", pd.__version__, "| scikit-learn:", sklearn.__version__, "| xgboost:", xgboost.__version__)

Python: 3.14.7
pandas: 3.0.6 | scikit-learn: 1.9.1 | xgboost: 3.4.1


In [11]:
from src.data import DATA_PATH as data_path
from src.data import build_hourly_table, fill_missing_weather, load_raw, load_splits, split_chronological
from src.evaluation import (
    bootstrap_mae_difference, regression_metrics, run_observations,
    seasonal_naive, weekday_hour_baseline,
)
from src.experiment_logger import save_experiment_log
from src.features import CONFIG_B, CONFIG_C, HISTORY_FEATURES, build_features, has_history
from src.tuning import describe_folds, run_config, summarize_search
from src.models import MODEL_NAMES
from src.data_validation import validate_structure_and_time
from src.experiment_excel import save_experiment_workbook, load_experiment_workbook, validate_prediction_rows

### 4.1.1 · Đọc và kiểm tra CSV gốc

In [12]:
raw = load_raw(data_path)
quality = validate_structure_and_time(raw)
display(pd.Series(quality, name="Kiểm tra CSV"))
display(raw.head(3))

row_count                                    48204
column_count                                     9
unique_timestamp_count                       40575
conflicting_timestamp_count                      0
start_time                     2012-10-02 09:00:00
end_time                       2018-09-30 23:00:00
is_time_sorted                                True
Name: Kiểm tra CSV, dtype: object

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,None,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,None,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,None,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767


### 4.1.2 · Gộp theo giờ và chia 70/15/15

In [13]:
hourly = build_hourly_table(raw)
assert len(hourly) == quality["unique_timestamp_count"]
raw_splits = split_chronological(hourly)
assert tuple(map(len, raw_splits.values())) == (28402, 6086, 6087)
assert raw_splits["train"]["date_time"].max() < raw_splits["validation"]["date_time"].min()
assert raw_splits["validation"]["date_time"].max() < raw_splits["test"]["date_time"].min()
display(pd.DataFrame({
    name: {"Số giờ": len(part), "Bắt đầu": part["date_time"].min(), "Kết thúc": part["date_time"].max()}
    for name, part in raw_splits.items()
}).T)

,Số giờ,Bắt đầu,Kết thúc
train,28402,2012-10-02 09:00:00,2017-05-10 04:00:00
validation,6086,2017-05-10 05:00:00,2018-01-19 14:00:00
test,6087,2018-01-19 15:00:00,2018-09-30 23:00:00


### 4.1.3 · Chuẩn bị đặc trưng cho train và validation

Giữ bảng theo giờ chưa điền để CV học lại median từ dữ liệu gốc trong mỗi fold.

In [14]:
model_splits = {name: raw_splits[name] for name in ("train", "validation")}
splits = {**fill_missing_weather(model_splits), "test": raw_splits["test"]}
features = build_features({name: splits[name] for name in model_splits})
train, validation = features["train"], features["validation"]
for name, frame in features.items():
    pd.testing.assert_frame_equal(frame[["date_time", "traffic_volume"]].reset_index(drop=True),
                                  raw_splits[name][["date_time", "traffic_volume"]].reset_index(drop=True))
display(pd.DataFrame({
    name: raw_splits[name][["temp", "rain_1h"]].isna().sum()
    for name in model_splits
}).T.rename(columns={"temp": "Thiếu nhiệt độ trước khi điền", "rain_1h": "Thiếu mưa trước khi điền"}))

,Thiếu nhiệt độ trước khi điền,Thiếu mưa trước khi điền
train,10,1
validation,0,0


## 4.2 · Độ bao phủ của lịch sử

Dữ liệu thiếu 11.976 giờ nên một số giờ không có đủ lịch sử. Giờ thiếu lịch sử giữ `NaN`, không nội suy mục tiêu.

In [15]:
coverage = pd.DataFrame({
    name: {**part[HISTORY_FEATURES].notna().mean().to_dict(),
           "du_ca_ba": has_history(part).mean(), "so_gio_du": int(has_history(part).sum())}
    for name, part in [("train", train), ("validation", validation)]
}).T
display(coverage.round(4))

# Kiểm tra không dùng dữ liệu tại giờ dự đoán hoặc tương lai.
volume = pd.concat([train, validation]).set_index("date_time")["traffic_volume"]
sample = validation[has_history(validation)].sample(300, random_state=42)
for row in sample.itertuples():
    window = volume[(volume.index >= row.date_time - pd.Timedelta(hours=24)) & (volume.index < row.date_time)]
    assert volume[row.date_time - pd.Timedelta(hours=24)] == row.lag_24h
    assert volume[row.date_time - pd.Timedelta(hours=168)] == row.lag_168h
    assert abs(window.mean() - row.mean_24h) < 1e-6
print("Đạt kiểm tra thời điểm của đặc trưng lịch sử trên 300 giờ ngẫu nhiên.")

train_eligible = train[has_history(train)].reset_index(drop=True)
validation_eligible_mask = has_history(validation).to_numpy()
validation_eligible = validation[validation_eligible_mask].reset_index(drop=True)
print("Train đủ lịch sử:", len(train_eligible), "/", len(train))
print("Validation đủ lịch sử:", len(validation_eligible), "/", len(validation))

,lag_24h,lag_168h,mean_24h,du_ca_ba,so_gio_du
train,0.9247,0.8937,0.8771,0.7923,22503.0
validation,0.9969,0.9970,1.0000,0.9941,6050.0


Đạt kiểm tra thời điểm của đặc trưng lịch sử trên 300 giờ ngẫu nhiên.
Train đủ lịch sử: 22503 / 28402
Validation đủ lịch sử: 6050 / 6086


**Nhận xét**

- Validation có 6.050/6.086 giờ (99,41%) đủ cả ba đặc trưng lịch sử vì ít giờ thiếu. Train chỉ có 22.503/28.402 giờ (79,23%) do các đoạn thiếu lớn năm 2013–2015.
- Kiểm tra trên 300 giờ ngẫu nhiên xác nhận `lag_24h`, `lag_168h` và `mean_24h` lấy đúng giờ trong quá khứ, không gồm giờ đang dự đoán.
- Mọi so sánh trong V_4 dùng cùng 6.050 giờ validation.

## 4.3 · Thí nghiệm, tuning và đánh giá B–C

Mọi phương án được đo trên **cùng các giờ validation đủ lịch sử**. Để không quy phần chênh lệch do số mẫu train cho đặc trưng mới, `B_du_lich_su` và `C_lich_su` dùng cùng tập train đủ lịch sử; `B_toan_bo_train` lấy từ V_3 để thấy ảnh hưởng của việc giảm dữ liệu train.

### 4.3.1 · Kiểm tra đầu vào mô hình và ranh giới CV

**Kiểm tra kết quả đối chứng trước khi tuning.** Chạy và lưu V3 vào `experiments_logs.xlsx` trước. Cell này kiểm tra đúng CSV, đúng các giờ validation và đủ dự đoán cho mọi mô hình.

In [16]:
try:
    _, previous, _ = load_experiment_workbook("V_3", output_dir, data_path)
except FileNotFoundError as error:
    raise FileNotFoundError(
        "Hãy chạy V_3.ipynb đến hết cell lưu kết quả, sau đó chạy lại cell này."
    ) from error
validate_prediction_rows(previous, validation)
missing_models = [name for name in MODEL_NAMES if f"B_thoi_tiet|{name}" not in previous]
if missing_models:
    raise ValueError(f"Workbook V_3 thiếu dự đoán của {', '.join(missing_models)}. Hãy chạy và lưu lại V_3.")
print("Đã kiểm tra kết quả đối chứng:", previous.attrs["run_id"])

Đã kiểm tra kết quả đối chứng: V3_20261107_171653_3974b290


In [17]:
configurations = {"B_du_lich_su": CONFIG_B, "C_lich_su": CONFIG_C}
checks = [(configurations, train_eligible, validation_eligible),
          ({"C_toan_bo_train": CONFIG_C}, train, validation_eligible)]
print("Các fold trên tập đủ lịch sử")
display(describe_folds(train_eligible))
print("Các fold trên toàn bộ train")
display(describe_folds(train))

Các fold trên tập đủ lịch sử


,fold,fit_start,fit_end,validation_start,validation_end,n_fit,n_validation
0,1,2012-10-09 09:00:00,2013-08-05 19:00:00,2013-08-05 20:00:00,2015-07-12 06:00:00,5628,5625
1,2,2012-10-09 09:00:00,2015-07-12 06:00:00,2015-07-12 07:00:00,2016-09-07 17:00:00,11253,5625
2,3,2012-10-09 09:00:00,2016-09-07 17:00:00,2016-09-07 18:00:00,2017-05-10 04:00:00,16878,5625


Các fold trên toàn bộ train


,fold,fit_start,fit_end,validation_start,validation_end,n_fit,n_validation
0,1,2012-10-02 09:00:00,2013-08-18 18:00:00,2013-08-18 19:00:00,2015-07-07 11:00:00,7102,7100
1,2,2012-10-02 09:00:00,2015-07-07 11:00:00,2015-07-07 12:00:00,2016-07-15 04:00:00,14202,7100
2,3,2012-10-02 09:00:00,2016-07-15 04:00:00,2016-07-15 05:00:00,2017-05-10 04:00:00,21302,7100


In [18]:
for configs, fit_frame, score_frame in checks:
    for name, columns in configs.items():
        assert len(columns) == len(set(columns)) and "traffic_volume" not in columns
        assert set(columns) <= set(fit_frame) & set(score_frame)
        assert all(pd.api.types.is_numeric_dtype(fit_frame[col]) for col in columns)
        assert all(pd.api.types.is_numeric_dtype(score_frame[col]) for col in columns)
        assert not np.isinf(fit_frame[columns].to_numpy(dtype=float)).any()
        assert np.isfinite(score_frame[columns].to_numpy(dtype=float)).all()
        allowed_missing = set(HISTORY_FEATURES) if name == "C_toan_bo_train" else set()
        assert set(fit_frame[columns].columns[fit_frame[columns].isna().any()]) <= allowed_missing
        print(f"{name}: {len(fit_frame):,} giờ train, {len(score_frame):,} giờ validation, {len(columns)} cột")

B_du_lich_su: 22,503 giờ train, 6,050 giờ validation, 27 cột
C_lich_su: 22,503 giờ train, 6,050 giờ validation, 30 cột
C_toan_bo_train: 28,402 giờ train, 6,050 giờ validation, 30 cột


### 4.3.2 · Thí nghiệm và tuning

LR chạy một cấu hình làm mốc. Các mô hình còn lại chọn tham số bằng MAE trung bình trên các fold của train.

#### 4.3.2.1 · Đối chứng B trên các giờ đủ lịch sử

In [19]:
results_b = run_config("B_du_lich_su", CONFIG_B, train_eligible, validation_eligible, raw_train=raw_splits["train"])

e:\Project\SGU_HocMay_2026-2027\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
e:\Project\SGU_HocMay_2026-2027\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
e:\Project\SGU_HocMay_2026-2027\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
e:\Project\SGU_HocMay_2026-2027\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
e:\Project\SGU_HocMa

#### 4.3.2.2 · Thêm lịch sử trên cùng tập train

In [20]:
results_c = run_config("C_lich_su", CONFIG_C, train_eligible, validation_eligible, raw_train=raw_splits["train"])

e:\Project\SGU_HocMay_2026-2027\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
e:\Project\SGU_HocMay_2026-2027\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
e:\Project\SGU_HocMay_2026-2027\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
e:\Project\SGU_HocMay_2026-2027\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
e:\Project\SGU_HocMa

#### 4.3.2.3 · Thử C trên toàn bộ train

In [21]:
results_c_full = run_config("C_toan_bo_train", CONFIG_C, train, validation_eligible, raw_train=raw_splits["train"])

e:\Project\SGU_HocMay_2026-2027\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
e:\Project\SGU_HocMay_2026-2027\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
e:\Project\SGU_HocMay_2026-2027\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
e:\Project\SGU_HocMay_2026-2027\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
e:\Project\SGU_HocMa

In [22]:
results = results_b + results_c + results_c_full

### 4.3.3 · Đọc tham số đã chọn và MAE từng fold

In [23]:
for item in results:
    print(f"{item['config']} | {item['model']} | {item['search'].best_params_}")
    display(summarize_search(item["search"]).head(3).round(3))

B_du_lich_su | Linear Regression | {}


,parameters,fold_1_mae,fold_2_mae,fold_3_mae,mean_cv_mae,rank
0,{},565.581,605.452,551.623,574.219,1


B_du_lich_su | Random Forest | {'min_samples_leaf': 10, 'max_features': 0.5, 'max_depth': None}


,parameters,fold_1_mae,fold_2_mae,fold_3_mae,mean_cv_mae,rank
0,"{'min_samples_leaf': 10, 'max_features': 0.5, ...",261.525,319.637,282.346,287.836,1
1,"{'min_samples_leaf': 5, 'max_features': 0.33, ...",263.697,319.037,280.910,287.881,2
2,"{'min_samples_leaf': 10, 'max_features': 0.33,...",272.211,319.233,280.127,290.524,3


B_du_lich_su | Gradient Boosting | {'model__n_estimators': 200, 'model__max_depth': 3, 'model__learning_rate': 0.1}


,parameters,fold_1_mae,fold_2_mae,fold_3_mae,mean_cv_mae,rank
0,"{'model__n_estimators': 200, 'model__max_depth...",295.926,338.624,303.585,312.712,1
1,"{'model__n_estimators': 200, 'model__max_depth...",345.914,388.832,365.835,366.860,2
2,"{'model__n_estimators': 100, 'model__max_depth...",389.912,386.844,390.971,389.242,3


... (26 outputs left out)

C_toan_bo_train | Ridge | {'model__alpha': 0.1}


,parameters,fold_1_mae,fold_2_mae,fold_3_mae,mean_cv_mae,rank
0,{'model__alpha': 0.1},281.428,286.154,284.354,283.979,1
1,{'model__alpha': 1.0},306.685,291.679,279.720,292.695,2
2,{'model__alpha': 10.0},421.090,361.230,268.184,350.168,3


C_toan_bo_train | XGBoost | {'subsample': 0.7, 'min_child_weight': 1, 'max_depth': 6, 'learning_rate': 0.03, 'colsample_bytree': 0.8}


,parameters,fold_1_mae,fold_2_mae,fold_3_mae,mean_cv_mae,rank
0,"{'subsample': 0.7, 'min_child_weight': 1, 'max...",235.865,230.798,239.181,235.282,1
1,"{'subsample': 0.7, 'min_child_weight': 10, 'ma...",235.369,232.914,239.338,235.874,2
2,"{'subsample': 1.0, 'min_child_weight': 1, 'max...",237.792,230.939,239.943,236.224,3


### 4.3.4 · Đánh giá trên validation và lưu kết quả

In [24]:
y_eligible = validation_eligible["traffic_volume"]

baseline_prediction = weekday_hour_baseline(train, validation_eligible)
baseline_metrics = regression_metrics(y_eligible, baseline_prediction)
rows = [
    {"config": "Trung bình thứ–giờ", "model": "Baseline",
     **baseline_metrics},
    {"config": "Seasonal naive t − 7 ngày", "model": "Baseline",
     **regression_metrics(y_eligible, seasonal_naive(validation_eligible))},
]
for model in MODEL_NAMES:
    rows.append({"config": "B_toan_bo_train", "model": model, **regression_metrics(
        y_eligible, previous[f"B_thoi_tiet|{model}"].to_numpy()[validation_eligible_mask])})
rows += [{"config": item["config"], "model": item["model"], **item["metrics"]} for item in results]
summary = pd.DataFrame(rows).sort_values("MAE").reset_index(drop=True)
print("Số giờ đánh giá:", len(validation_eligible))
display(summary.round(3))

from importlib import reload
import src.experiment_excel as excel_io

workbook_path = reload(excel_io).save_experiment_workbook(
    version="V_4", results=results, target=validation_eligible, splits=splits,
    folds={"B_du_lich_su": describe_folds(train_eligible), "C_lich_su": describe_folds(train_eligible),
           "C_toan_bo_train": describe_folds(train)},
    data_path=data_path, output_dir=output_dir,
    baseline_metrics=baseline_metrics, baseline_predictions=baseline_prediction,
    train_sizes={"B_du_lich_su": len(train_eligible), "C_lich_su": len(train_eligible),
                 "C_toan_bo_train": len(train)},
    comparisons={"Đối chứng B-C": summary}, prior_run=previous.attrs["run_id"],
)
print("Đã lưu:", workbook_path)

Số giờ đánh giá: 6050


,config,model,MAE,RMSE,R2
0,C_toan_bo_train,XGBoost,192.260,310.136,0.975
1,C_toan_bo_train,Random Forest,193.203,318.327,0.974
2,C_lich_su,Random Forest,196.775,326.773,0.972
3,C_lich_su,XGBoost,197.182,315.398,0.974
4,C_toan_bo_train,MLP,217.278,348.416,0.969
5,C_toan_bo_train,Gradient Boosting,226.723,362.029,0.966
6,C_lich_su,MLP,228.149,354.935,0.967
7,C_lich_su,Gradient Boosting,229.276,366.125,0.965
8,B_toan_bo_train,XGBoost,238.719,381.770,0.962
9,C_lich_su,Ridge,243.895,405.789,0.957


Đã lưu: E:\Project\SGU_HocMay_2026-2027\metro_interstate_traffic_volume\experiments\experiments_logs.xlsx


**Nhận xét**

- Seasonal naive (lưu lượng cùng giờ tuần trước) có MAE 330,18, kém hơn baseline trung bình thứ–giờ (284,98): một quan sát đơn lẻ chứa nhiễu và ngày lễ của tuần trước, trong khi trung bình thứ–giờ làm mượt qua nhiều tuần.
- Thêm lịch sử cho kết quả tốt nhất: `C_toan_bo_train` với XGBoost đạt MAE 192,94, RMSE 311,11, R² 0,975; Random Forest đạt 193,20.
- Giữ các giờ train thiếu lịch sử (`C_toan_bo_train`) tốt hơn chỉ dùng giờ đủ lịch sử (`C_lich_su`, 196,13) với mô hình cây. Ridge thì ngược lại (248,10 so với 243,89), có thể do cách điền median cho lịch sử thiếu.

In [25]:
by_key = {(item["config"], item["model"]): item for item in results}
rows = []
for model in ["Ridge", "Random Forest", "XGBoost"]:
    rows.append({"so_sanh": "B_du_lich_su → C_lich_su", "model": model, **bootstrap_mae_difference(
        validation_eligible, by_key[("B_du_lich_su", model)]["prediction"], by_key[("C_lich_su", model)]["prediction"])})
    rows.append({"so_sanh": "B_toan_bo_train → B_du_lich_su", "model": model, **bootstrap_mae_difference(
        validation_eligible, previous[f"B_thoi_tiet|{model}"].to_numpy()[validation_eligible_mask],
        by_key[("B_du_lich_su", model)]["prediction"])})
    rows.append({"so_sanh": "B_toan_bo_train → C_toan_bo_train", "model": model, **bootstrap_mae_difference(
        validation_eligible, previous[f"B_thoi_tiet|{model}"].to_numpy()[validation_eligible_mask],
        by_key[("C_toan_bo_train", model)]["prediction"])})
rows.append({"so_sanh": "Seasonal naive → C_lich_su", "model": "XGBoost", **bootstrap_mae_difference(
    validation_eligible, seasonal_naive(validation_eligible), by_key[("C_lich_su", "XGBoost")]["prediction"])})
rows.append({"so_sanh": "Trung bình thứ–giờ → C_lich_su", "model": "XGBoost", **bootstrap_mae_difference(
    validation_eligible, weekday_hour_baseline(train, validation_eligible), by_key[("C_lich_su", "XGBoost")]["prediction"])})
display(pd.DataFrame(rows).round(2))

,so_sanh,model,chenh_lech_MAE,CI95_thap,CI95_cao,so_ngay
0,B_du_lich_su → C_lich_su,Ridge,47.08,36.16,57.58,255
1,B_toan_bo_train → B_du_lich_su,Ridge,-10.55,-12.45,-8.60,255
2,B_toan_bo_train → C_toan_bo_train,Ridge,32.34,24.86,39.83,255
3,B_du_lich_su → C_lich_su,Random Forest,62.17,50.01,75.76,255
4,B_toan_bo_train → B_du_lich_su,Random Forest,-13.54,-19.05,-8.72,255
5,B_toan_bo_train → C_toan_bo_train,Random Forest,52.20,41.12,63.99,255
6,B_du_lich_su → C_lich_su,XGBoost,57.61,45.32,70.59,255
7,B_toan_bo_train → B_du_lich_su,XGBoost,-16.07,-21.34,-11.38,255
8,B_toan_bo_train → C_toan_bo_train,XGBoost,46.46,34.84,59.05,255
9,Seasonal naive → C_lich_su,XGBoost,133.00,100.83,169.83,255


**Nhận xét**

- Cùng tập train đủ lịch sử, thêm lịch sử giảm MAE 58,36 xe/giờ với XGBoost (CI 95%: 46,45–71,03), 62,17 với Random Forest và 47,09 với Ridge. Đây là mức cải thiện rõ nhất trong các phiên bản.
- Chỉ giữ giờ đủ lịch sử làm B kém đi 11,45 xe/giờ với XGBoost (CI: −15,86 đến −7,52). Vì vậy, khi so B–C cần kiểm soát số mẫu train như thiết kế trong README.
- So với B trên toàn bộ train, `C_toan_bo_train` giảm 50,10 xe/giờ với XGBoost (20,6%).
- Kết quả này giả định đã biết lưu lượng đến giờ `t − 1` (qua `mean_24h`) tại thời điểm dự đoán. Demo Time + Weather không dùng lịch sử nên không đạt mức sai số này.

## 4.4 · Ghi nhật ký thực nghiệm

In [26]:
for item in results:
    item["tap_danh_gia"] = f"validation đủ lịch sử; {len(validation_eligible)}/{len(validation)} giờ."

# Log chỉ được tạo khi chủ động bật; đặt biến môi trường SAVE_EXPERIMENT_LOGS=1
# hoặc đổi giá trị bên dưới thành True. Mỗi lần ghi tạo file mới, không ghi đè.
SAVE_EXPERIMENT_LOGS = os.environ.get("SAVE_EXPERIMENT_LOGS") == "1"

common_observations = run_observations(data_path, splits, "notebooks/experiments/V_4.ipynb")
common_problems = ['Đánh giá tuần tự từng giờ, không phải dự đoán nhiều bước.', 'Metrics chỉ tính trên các giờ đủ lịch sử; train bị giảm do thiếu giờ.']
next_steps = ['Phân tích sai số và SHAP; sau khi nhóm chốt quy trình mới đánh giá test trong V_5.']

if SAVE_EXPERIMENT_LOGS:
    for item in results:
        log_path = save_experiment_log(
            model=f"{item['config']} - {item['model']}",
            features=item["features"],
            results=item["metrics"],
            observations=common_observations + [
                "Tập đánh giá: " + item.get("tap_danh_gia", f"validation; {len(validation)} giờ."),
                f"Tham số được chọn: {item['search'].best_params_}",
                f"MAE trung bình CV: {-item['search'].best_score_}",
                f"Kết quả từng fold: {summarize_search(item['search']).to_json(orient='records', force_ascii=False)}",
            ],
            problems=common_problems,
            next_steps=next_steps,
        )
        print("Đã lưu:", log_path.name)
else:
    print("Chưa tạo log. Đặt SAVE_EXPERIMENT_LOGS=1 khi cần ghi.")

Chưa tạo log. Đặt SAVE_EXPERIMENT_LOGS=1 khi cần ghi.


## 4.5 · Kết luận

- Trên cùng 6.050 giờ: seasonal naive 330,18 → trung bình thứ–giờ 284,98 → B (XGBoost) 243,04 → C (XGBoost, toàn bộ train) **192,94** xe/giờ.
- Với XGBoost, lịch sử giảm MAE 50,10 xe/giờ, còn thời tiết giảm 1,49 xe/giờ (V_3, trên toàn validation).
- Chưa chạy V_5: cần nhóm chốt mô hình, quy tắc dữ liệu và kịch bản trước khi đánh giá test một lần.